## The Face Recognition Exam System

<hr>

## Introduction:

The Face Recognition Exam System is a comprehensive and innovative solution designed to simplify, secure, and streamline the examinations' attendance and seat allocation process. Incorporating state-of-the-art artificial intelligence and facial recognition capabilities, this system ensures that only authenticated students are granted access. By leveraging computer vision techniques and machine learning models, it offers seamless enrollment, recognition, and real-time decision-making – all in one integrated platform.

<hr>

In [1]:
!pip install opencv-python
!pip install numpy
!pip install scikit-learn
!pip install keras
!pip install tensorflow
!apt-get update && apt-get install -y libgl1
!pip install mtcnn
!pip install keras_facenet
!pip install deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 25.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Get:1 http://deb.debian.org/debian bullseye InRelease [116 kB]
Get:2 http://deb.debian.org/debian-security bullseye-security InRelease [27.2 kB]
Get:3 http://deb.debian.org/debian bullseye-updates InRelease [44.1 kB]
Get:4 http://deb.debian.org/debian bullseye/main amd64 Packages [8,066 kB]
Get:5 http://deb.debian.org/debian-security bullseye-security/

#### We start by Importing the required libraries. The following libraries are used in this project:

In [ ]:
import cv2
import numpy as np
import os
import joblib
import pandas as pd
from datetime import datetime
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from mtcnn import MTCNN
from keras_facenet import FaceNet
from deepface import DeepFace
from datetime import datetime

In [103]:
facenet_model = FaceNet()
mask_detector_model_path = '/work/Models/mask_detector.model'
mask_detector_model = load_model(mask_detector_model_path)
svm_model_path = "/work/Models/svm_model.pkl"
label_encoder_path = "/work/Models/label_encoder.pkl"
attendance_file = "/work/Attendance/exam_attendance.csv"

if os.path.exists(svm_model_path) and os.path.exists(label_encoder_path):
    svm_model = joblib.load(svm_model_path)
    label_encoder = joblib.load(label_encoder_path)
else:
    svm_model = SVC(kernel='linear', probability=True)
    label_encoder = LabelEncoder()

detector = MTCNN()

def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Error: Image file not found at {image_path}")
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

def extract_features(image_path):
    processed_image = preprocess_image(image_path)
    embeddings = facenet_model.embeddings([processed_image])
    return embeddings[0]

def enroll_students(student_images, student_ids):
    features = []
    for image_path in student_images:
        features.append(extract_features(image_path))
    features = np.array(features)

    if len(student_ids) > 0:
        labels = label_encoder.fit_transform(student_ids)
        svm_model.fit(features, labels)
        joblib.dump(svm_model, svm_model_path)
        joblib.dump(label_encoder, label_encoder_path)
        print("Student data enrolled successfully!")

def authenticate_student(image_path, confidence_threshold=0.6):
    """Authenticates a student image with security confidence threshold check."""
    if not hasattr(svm_model, "support_"):
        print("Authentication error: SVM model is not trained.")
        return None
    features = extract_features(image_path).reshape(1, -1)
    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(features)[0]
        max_prob = np.max(probs)
        best_class = np.argmax(probs)
        if max_prob < confidence_threshold:
            print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
            return None
        return label_encoder.inverse_transform([best_class])[0]
    else:
        prediction = svm_model.predict(features)
        return label_encoder.inverse_transform(prediction)[0]

# live-web-cam function
# def authenticate_student_from_frame(face_img, confidence_threshold=0.6):
#     """Authenticates a student frame with security confidence threshold check."""
#     if not hasattr(svm_model, "support_"):
#         print("Authentication error: SVM model is not trained.")
#         return None
#     face_img_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
#     face_img_resized = cv2.resize(face_img_rgb, (160, 160))  # FaceNet input size
#     embeddings = facenet_model.embeddings([face_img_resized])
#     if hasattr(svm_model, "predict_proba"):
#         probs = svm_model.predict_proba(embeddings)[0]
#         max_prob = np.max(probs)
#         best_class = np.argmax(probs)
#         if max_prob < confidence_threshold:
#             print(f"Authentication failed: Low confidence score ({max_prob:.2f} < {confidence_threshold}).")
#             return None
#         return label_encoder.inverse_transform([best_class])[0]
#     else:
#         prediction = svm_model.predict(embeddings)
#         return label_encoder.inverse_transform(prediction)[0]
def detect_faces(frame):

    faces = detector.detect_faces(frame)
    detected_faces = []

    for face in faces:
        if face['confidence'] > 0.9:
            x, y, w, h = face['box']
            face_crop = frame[y:y+h, x:x+w]
            
            try:
                face_crop_resized = cv2.resize(face_crop, (224, 224))
                analysis = DeepFace.analyze(face_crop, actions=["gender"], enforce_detection=False)
                if isinstance(analysis, list) and len(analysis) > 0:
                    detected_gender = analysis[0].get("dominant_gender", "Unknown")
                else:
                    detected_gender = "Unknown"
            except Exception as e:
                detected_gender = "Unknown"

            detected_faces.append((x, y, w, h, detected_gender))

    return detected_faces

# def detect_faces(frame):
#     faces = detector.detect_faces(frame)
#     detected_faces = []
#     for face in faces:
#         if face['confidence'] > 0.9:
#             x, y, w, h = face['box']
#             detected_faces.append((x, y, w, h))
#     return detected_faces


def detect_and_log_mask(face_img):
    try:
        original_face_img = face_img.copy()
        face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        face_img_resized = cv2.resize(face_img, (224, 224))

        # Apply preprocessing specific to mask detection model
        preprocessed_img = img_to_array(face_img_resized)
        preprocessed_img = preprocess_input(preprocessed_img)
        preprocessed_img = np.expand_dims(preprocessed_img, axis=0)

        # Perform prediction with the mask detector model
        mask_prob = mask_detector_model.predict(preprocessed_img, verbose=0)[0]

        mask_detected = mask_prob[0] > 0.3
        mask_status = "Mask" if mask_detected else "No Mask"

        #print(f"Mask Probability: {mask_prob[0]*100:.2f}%, No Mask Probability: {mask_prob[1]*100:.2f}%")

        return mask_status, mask_detected

    except Exception as e:
        print(f"Error during mask detection: {e}")
        return "Error", False


def get_occupied_seats():
    occupied_seats = set()
    if os.path.exists(attendance_file):
        attendance_data = pd.read_csv(attendance_file)
        for _, row in attendance_data.iterrows():
            occupied_seats.add((int(row['Exam Hall'].split('-')[1]), int(row['Seat ID'].split('-')[1])))
    return occupied_seats

def allocate_seat(course=None):
    available_seats = [(r, s) for r in range(1, 6) for s in range(1, 31)]  # Example hall layout
    occupied_seats = get_occupied_seats()
    free_seats = [seat for seat in available_seats if seat not in occupied_seats]

    if not free_seats:
        return None
    if course:
        free_seats = sorted(free_seats, key=lambda seat: seat[0])  
    else:
        np.random.shuffle(free_seats)

    return free_seats[0]

def mark_attendance(student_id, seat, gender="Unknown"):
    if not seat:
        return "No seats available!"

    now = datetime.now()
    date = now.strftime("%d-%m-%y")
    time = now.strftime("%H:%M")

    # Unpack seat information at the very beginning
    exam_hall, seat_id = seat

    if not os.path.exists("/work/Attendance"):
        os.makedirs("/work/Attendance")

    if not os.path.exists(attendance_file):
        df = pd.DataFrame(columns=['Date', 'Time', 'Student ID', 'Gender', 'Status', 'Exam Hall', 'Seat ID'])
        df.to_csv(attendance_file, index=False)

    if os.path.exists(attendance_file):
        existing_data = pd.read_csv(attendance_file)
        if ((existing_data["Student ID"] == student_id) & (existing_data["Exam Hall"] == f"R-{exam_hall}") &
                (existing_data["Seat ID"] == f"S-{seat_id}")).any():
            return f"Attendance already marked for Student ID: {student_id} in Room: R-{exam_hall}, Seat: S-{seat_id}."

    attendance_status = "Present"
    data = {
        "Date": date,
        "Time": time,
        "Student ID": student_id,
        "Gender": gender,
        "Status": attendance_status,
        "Exam Hall": f"R-{exam_hall}",
        "Seat ID": f"S-{seat_id}"
    }

    df = pd.DataFrame([data])
    if os.path.exists(attendance_file):
        df.to_csv(attendance_file, mode='a', header=False, index=False)
    else:
        df.to_csv(attendance_file, mode='w', header=True, index=False)

    return f"Seat Allocated: Gender:-{gender} Room:-{exam_hall} Seat:-{seat_id} | Attendance Marked: {attendance_status} on {date} at {time}"

def general_function(face_image, student_images=None, student_ids=None):
    try:
        processed_frame = preprocess_image(face_image)
        detected_faces = detect_faces(processed_frame)
        if not detected_faces:
            print("No faces detected in the image!")
            return None

        print(f"Step 0: Number of faces detected: {len(detected_faces)}")
        x, y, w, h, detected_gender = detected_faces[0]
        print(f"Detected gender: {detected_gender}")

        mask_status, is_mask_detected = detect_and_log_mask(processed_frame)
        print(f"Step 1: Mask Status: {mask_status}")

        if is_mask_detected:
            print(f"Mask detected! Student cannot be authenticated.")
            cv2.putText(processed_frame, "Mask Detected! Remove mask", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            return None

        if student_images and student_ids:
            enroll_students(student_images, student_ids)

        recognized_id = authenticate_student(face_image)
        if recognized_id is None:
            print("Authentication failed: No matching student found.")
            return None

        seat_info = allocate_seat()
        if seat_info is None:
            print("No seats available!")
            return None
        exam_hall, seat_id = seat_info

        mark_attendance(recognized_id, seat_info, detected_gender)

        print(f"Student ID: {recognized_id}, {seat_info}")
        return recognized_id, seat_info

    except FileNotFoundError as e:
        print(f"Error: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error encountered: {e}")
        return None

if __name__ == "__main__":
    print("Face Recognition Exam System Loaded and Ready.")

Face Recognition Exam System Loaded and Ready.


<hr>

## Feature

- Face Detection and Gender Recognition

- Real-Time Mask Detection

- Student Authentication System

- Seat Allocation System

- Attendance Logging

- Enrollment Functionality

- Error Handling and Feedback

- Low Resource Dependency

<hr>

## Prompting or Function Calling 

In [105]:
face_image = "/work/Student_img/student2.jpg"

result = general_function(face_image)

if result:
    recognized_id, seat_info = result
    print(f"✅ Recognized Student ID: {recognized_id}, {seat_info}")
else:
    print("❌ No student recognized or attendance failed.")

Step 0: Number of faces detected: 1
Detected gender: Woman
Mask Probability: 0.75%, No Mask Probability: 99.25%
Step 1: Mask Status: No Mask
1/1 [==============================] - 3s 3s/step
Student ID: ID_002, (5, 19)
✅ Recognized Student ID: ID_002, (5, 19)


In [ ]:
#TODO Add more features
    """
    Function to mark the attendance of a student with enhanced features:
    - Includes course information.
    - Adds a timestamp of attendance marking.
    - Validates if the student is already marked for the session.
    
    Parameters:
    student_id (str): ID of the student.
    seat (tuple): Tuple containing exam hall and seat id.
    gender (str): Gender of the student. Default is "Unknown".
    course (str): Course name or code related to the student. Default is "Unknown".
    timestamp (datetime): Timestamp of attendance. If None, current time is used.

    Returns:
    str: Status of attendance marking.
    """

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=656a6e4b-42df-434c-8df0-b6590b48d22a' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>